# Uloha 2 - ASCII Art
Neuronove siete na rozpoznavanie znakov a konverzia obrazkov na ASCII text.

**Autor:** Ostapchuk

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image

# seed - aby vysledky boli vzdy rovnake
torch.manual_seed(42)
np.random.seed(42)

print("PyTorch verzia:", torch.__version__)

In [ ]:

# znaky, ktore rozpoznavame
# medzera je kodovana ako nuly (prazdny blok)
ZNAKY = [" ", "|", "/", "\\", "_", "-", "^", "o"]
POCET_ZNAKOV = len(ZNAKY)
SIRKA_BLOKU = 8
VYSKA_BLOKU = 14
VECTORSIZE = SIRKA_BLOKU * VYSKA_BLOKU



## Priprava datasetu
Rucne vytvorene binarne obrazky znakov (8x14 pixelov). Kazdy znak ma niekolko variant. Medzera je kodovana ako nuly.

In [ ]:
# [AI] programove generovanie binarnych sablon znakov (8x14 pixelov)
# namiesto manualneho zadavania poli pouzivame funkcie na kreslenie tvarov
# kazdy znak ma niekolko variant pre lepsiu generalizaciu
def _empty():
    return np.zeros((VYSKA_BLOKU, SIRKA_BLOKU), dtype=np.float32)
def _fill_row(img, row, x1, x2, val=1.0):
    img[row, max(0, x1):min(SIRKA_BLOKU, x2)] = val
def _fill_col(img, col, y1, y2, val=1.0):
    img[max(0, y1):min(VYSKA_BLOKU, y2), col] = val
def _fill_diag(img, direction, thickness=1, step=2):
    for y in range(VYSKA_BLOKU):
        col = y // step
        if direction == "/":
            x = SIRKA_BLOKU - 1 - col
        else:
            x = col
        for t in range(thickness):
            px = x + t
            if 0 <= px < SIRKA_BLOKU:
                img[y, px] = 1.0
def _fill_circle(img, cy, cx, radius, filled=True):
    for y in range(VYSKA_BLOKU):
        for x in range(SIRKA_BLOKU):
            dist = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
            if filled:
                if dist <= radius:
                    img[y, x] = 1.0
            else:
                if abs(dist - radius) < 0.8:
                    img[y, x] = 1.0
def _fill_triangle(img, top_y, top_x, base_width, height, val=1.0):
    for row in range(height):
        if top_y + row >= VYSKA_BLOKU:
            break
        half = (row + 1) * base_width / (2 * height)
        x1 = int(top_x - half)
        x2 = int(top_x + half + 0.5)
        _fill_row(img, top_y + row, x1, x2, val)
# kazdy znak ma niekolko geometrickych variant pre lepsiu generalizaciu
def vytvor_znakove_sablony():
    sablony = {}
    sablony[" "] = [_empty()]
    sablony["|"] = []
    for col in [4, 3, 5]:
        t = _empty()
        _fill_col(t, col, 0, VYSKA_BLOKU)
        sablony["|"].append(t)
    for col in [3]:
        t = _empty()
        _fill_col(t, col, 0, VYSKA_BLOKU)
        _fill_col(t, col + 1, 0, VYSKA_BLOKU)
        sablony["|"].append(t)
    sablony["/"] = []
    for thick in [1, 2]:
        t = _empty()
        _fill_diag(t, "/", thickness=thick)
        sablony["/"].append(t)
    sablony["\\"] = []
    for thick in [1, 2]:
        t = _empty()
        _fill_diag(t, "\\", thickness=thick)
        sablony["\\"].append(t)
    sablony["_"] = []
    for row in [13, 12]:
        t = _empty()
        _fill_row(t, row, 0, SIRKA_BLOKU)
        sablony["_"].append(t)
    sablony["-"] = []
    for row in [6, 5, 7]:
        t = _empty()
        _fill_row(t, row, 1, SIRKA_BLOKU - 1)
        sablony["-"].append(t)
    for row in [5]:
        t = _empty()
        _fill_row(t, row, 1, SIRKA_BLOKU - 1)
        _fill_row(t, row + 1, 1, SIRKA_BLOKU - 1)
        sablony["-"].append(t)
    sablony["^"] = []
    t = _empty()
    _fill_triangle(t, 0, 4, 8, 4)
    sablony["^"].append(t)
    t = _empty()
    _fill_row(t, 0, 4, 5)
    _fill_row(t, 1, 3, 6)
    _fill_row(t, 2, 2, 7)
    _fill_row(t, 3, 1, 8)
    sablony["^"].append(t)
    t = _empty()
    _fill_triangle(t, 0, 4, 8, 4)
    _fill_row(t, 1, 3, 6)
    sablony["^"].append(t)
    sablony["o"] = []
    for cy, r in [(7, 2.5), (6, 2.5), (6, 2.2)]:
        t = _empty()
        _fill_circle(t, cy, 3.5, r, filled=True)
        sablony["o"].append(t)
    t = _empty()
    _fill_circle(t, 6, 3.5, 2.5, filled=False)
    sablony["o"].append(t)
    return sablony

In [ ]:
# [AI] augmentacne funkcie (posun, sum) som navrhol s pomocou AI
# augmentacia zvacsuje dataset umelymi variantmi
def posun_obrazka(img, dx, dy):
    vysledok = np.zeros_like(img)
    for y in range(VYSKA_BLOKU):
        for x in range(SIRKA_BLOKU):
            ny, nx = y + dy, x + dx
            if 0 <= ny < VYSKA_BLOKU and 0 <= nx < SIRKA_BLOKU:
                vysledok[ny, nx] = img[y, x]
    return vysledok
def pridaj_sum(img, intensita=0.1):
    sum = np.random.uniform(-intensita, intensita, img.shape).astype(np.float32)
    return np.clip(img + sum, 0.0, 1.0)
# dataset obsahuje posunutia, sum a rotacie - realne obrazky su vzdy trochu ine
def vytvor_dataset(sablony, augmentacia=True):
    vstupy = []
    vystupy = []
    for idx, znak in enumerate(ZNAKY):
        one_hot = np.zeros(POCET_ZNAKOV, dtype=np.float32)
        one_hot[idx] = 1.0
        for sablona in sablony[znak]:
            flat = sablona.flatten()
            vstupy.append(flat)
            vystupy.append(one_hot)
            if augmentacia and znak != " ":
                for dx in [-2, -1, 1, 2]:
                    for dy in [-1, 1]:
                        posunuty = posun_obrazka(sablona, dx, dy)
                        vstupy.append(posunuty.flatten())
                        vystupy.append(one_hot)
                for _ in range(2):
                    zasumeny = pridaj_sum(sablona, 0.1)
                    vstupy.append(zasumeny.flatten())
                    vystupy.append(one_hot)
    vstupy_t = torch.tensor(np.array(vstupy), dtype=torch.float32)
    vystupy_t = torch.tensor(np.array(vystupy), dtype=torch.float32)
    return list(zip(vstupy_t, vystupy_t))

In [ ]:
sablony = vytvor_znakove_sablony()
print("Znaky:", ZNAKY)
print("Pocet sablon:", sum(len(v) for v in sablony.values()))
data = vytvor_dataset(sablony, augmentacia=True)
print(f"Dataset: {len(data)} vzoriek (s augmentaciou)")

In [ ]:
# Model 1: mensia siet
# 2 skryte vrstvy, malo parametrov
class NetSmall(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(VECTORSIZE, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, POCET_ZNAKOV)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        x = self.sigmoid(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        return x
# Model 2: stredna siet
# 3 skryte vrstvy, stredny pocet parametrov
class NetMedium(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(VECTORSIZE, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, POCET_ZNAKOV)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        x = self.sigmoid(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        x = self.sigmoid(self.fc4(x))
        return x
# Model 3: vacsia siet
# [AI] inspiracia pre hlbku siete z AI - vacsia kapacita pre zlozitejsie vzory
# 4 skryte vrstvy s ReLU, dropout pre regularizaciu
class NetLarge(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(VECTORSIZE, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 32)
        self.fc5 = nn.Linear(32, POCET_ZNAKOV)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.relu(self.fc4(x))
        x = self.sigmoid(self.fc5(x))
        return x

## Trenovanie a testovanie
V kazdej epoche sa nahodne zamiesaju vzorky. Pouzivame SSE chybu a SGD optimizer.

In [ ]:
# trenovanie siete
# v kazdej epoche sa nahodne zamiesaju vzorky
# pouzivame SSE chybu a SGD optimizer
def train(model, data, epochs, lr, print_every=500):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    historia = []
    for epoch in range(1, epochs + 1):
        poradie = torch.randperm(len(data))
        celkova_chyba = 0.0
        for i in poradie:
            vstup, ciel = data[i]
            vystup = model(vstup)
            chyba = torch.sum((vystup - ciel) ** 2)
            celkova_chyba += chyba.item()
            optimizer.zero_grad()
            chyba.backward()
            optimizer.step()
        historia.append(celkova_chyba)
        if epoch % print_every == 0 or epoch == epochs:
            print(f"Epocha {epoch}  Global error  {celkova_chyba:.5f}")
    return historia
# testovanie na trenovacich datach
# accuracy = kolko znakov bolo rozpoznanych spravne
# reliability = kolko vystupov je blizko 0 alebo 1
def test(model, data, epsilon=0.2):
    model.eval()
    spravne = 0
    celk_acc = 0
    celk_rel = 0
    print("\nTestovanie")
    print(f"{'Znak':<6}{'Predikcia':<10}{'Output':<30}{'Error':<10}{'Accuracy':<12}{'Reliability'}")
    print("-" * 100)
    with torch.no_grad():
        for vstup, ciel in data:
            vystup = model(vstup)
            pred_idx = torch.argmax(vystup).item()
            true_idx = torch.argmax(ciel).item()
            chyba = torch.sum((vystup - ciel) ** 2).item()
            acc = (torch.round(vystup) == ciel).sum().item() / len(ciel) * 100
            celk_acc += acc
            rel_ok = ((vystup < epsilon) | (vystup > 1 - epsilon)).sum().item()
            rel = rel_ok / len(vystup) * 100
            celk_rel += rel
            if pred_idx == true_idx:
                spravne += 1
            pred_znak = ZNAKY[pred_idx]
            true_znak = ZNAKY[true_idx]
            out_str = " ".join([f"{v:.2f}" for v in vystup])
            print(f"{true_znak!r:<6}{pred_znak!r:<10}{out_str:<30}{chyba:<10.3f}{acc:.0f}%{'':<8}{rel:.0f}%")
    n = len(data)
    print(f"\nSpravne: {spravne}/{n}")
    print(f"Priemerna accuracy: {celk_acc / n:.1f}%")
    print(f"Priemerna reliability: {celk_rel / n:.1f}%")
    model.train()
    return spravne, celk_acc / n, celk_rel / n

## Konverzia obrazka na ASCII
Rozrezanie obrazka na bloky a konverzia na ASCII text.

In [ ]:
# rozrezanie obrazka na bloky a konverzia na ASCII
# [AI] logiku rozrezania obrazka som navrhol s pomocou AI
def obrazok_na_ascii(model, cesta_obrazka, sirka_bloku=8, vyska_bloku=14):
    img = Image.open(cesta_obrazka).convert("L")
    img_array = np.array(img, dtype=np.float32) / 255.0
    vyska, sirka = img_array.shape
    bloky_x = sirka // sirka_bloku
    bloky_y = vyska // vyska_bloku
    ascii_vysledok = []
    model.eval()
    with torch.no_grad():
        for by in range(bloky_y):
            riadok = ""
            for bx in range(bloky_x):
                y1 = by * vyska_bloku
                y2 = y1 + vyska_bloku
                x1 = bx * sirka_bloku
                x2 = x1 + sirka_bloku
                blok = img_array[y1:y2, x1:x2]
                flat = torch.tensor(blok.flatten(), dtype=torch.float32)
                vystup = model(flat)
                idx = torch.argmax(vystup).item()
                riadok += ZNAKY[idx]
            ascii_vysledok.append(riadok)
    model.train()
    return "\n".join(ascii_vysledok)

## Model 1: NetSmall (112->64->32->8)
Mensia siet s 2 skrytymi vrstvami a Sigmoid aktivaciou. Maly pocet parametrov.

### Experiment 1.1
Zakladny experiment s lr=1.0 a 2000 epochami.

In [ ]:
print("\n=== Experiment 1.1 (lr=1.0, 2000 epoch) ===\n")
torch.manual_seed(42)
np.random.seed(42)
model_1_1 = NetSmall()
hist_1_1 = train(model_1_1, data, epochs=2000, lr=1.0, print_every=500)
s_1_1, a_1_1, r_1_1 = test(model_1_1, data)

**Diskusia 1.1:**
Mensia siet s lr=1.0 a 2000 epochami sa uci, ale chyba klesa pomaly. Mensia kapacita siete moze obmedzovat schopnost naucit sa vsetky znaky spravne.

### Experiment 1.2
Zvysenie learning rate na 2.0 a predlzenie treningu na 3000 epoch.

In [ ]:
print("\n=== Experiment 1.2 (lr=2.0, 3000 epoch) ===\n")
torch.manual_seed(42)
np.random.seed(42)
model_1_2 = NetSmall()
hist_1_2 = train(model_1_2, data, epochs=3000, lr=2.0, print_every=1000)
s_1_2, a_1_2, r_1_2 = test(model_1_2, data)

**Diskusia 1.2:**
Zvysenie lr na 2.0 a viac epoch vyrazne pomohlo. Vyssi learning rate umoznuje rychlejsie ucenie, ale moze byt menej stabilny.

### Experiment 1.3
Krokovy learning rate - na zaciatku velky lr (rychle ucenie) a postupne znizovanie (presnejsie doladenie).

In [ ]:
# [AI] napad krokoveho lr som nasiel s pomocou AI
print("\n=== Experiment 1.3 (krokovy lr) ===\n")
torch.manual_seed(42)
np.random.seed(42)
model_1_3 = NetSmall()
hist_1_3 = []
print("--- faza 1: lr=2.0, 1000 epoch ---")
h = train(model_1_3, data, epochs=1000, lr=2.0, print_every=500)
hist_1_3.extend(h)
print("\n--- faza 2: lr=0.5, 1000 epoch ---")
h = train(model_1_3, data, epochs=1000, lr=0.5, print_every=500)
hist_1_3.extend(h)
print("\n--- faza 3: lr=0.05, 500 epoch ---")
h = train(model_1_3, data, epochs=500, lr=0.05, print_every=250)
hist_1_3.extend(h)
s_1_3, a_1_3, r_1_3 = test(model_1_3, data)

**Diskusia 1.3:**
Krokovy learning rate (2.0 -> 0.5 -> 0.05) dava najlepsie vysledky pre NetSmall. Velky lr na zaciatku rychlo najde dobry smer, maly lr na konci presne doladi.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(hist_1_1, label="Exp 1.1 (lr=1.0)")
plt.plot(hist_1_2, label="Exp 1.2 (lr=2.0)")
plt.plot(hist_1_3, label="Exp 1.3 (krokovy lr)")
plt.xlabel("Epocha")
plt.ylabel("Global error")
plt.title("Model 1 (NetSmall) - Priebeh chyby")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.show()

## Model 2: NetMedium (112->128->64->32->8)
Stredna siet s 3 skrytymi vrstvami a Sigmoid aktivaciou. Stredny pocet parametrov.

### Experiment 2.1
Zakladny experiment s lr=1.0 a 2000 epochami.

In [ ]:
print("\n=== Experiment 2.1 (lr=1.0, 2000 epoch) ===\n")
torch.manual_seed(42)
np.random.seed(42)
model_2_1 = NetMedium()
hist_2_1 = train(model_2_1, data, epochs=2000, lr=1.0, print_every=500)
s_2_1, a_2_1, r_2_1 = test(model_2_1, data)

**Diskusia 2.1:**
Stredna siet s lr=1.0 dosahuje lepsie vysledky ako NetSmall pri rovnakom nastaveni. Viac parametrov = lepsia reprezentacna kapacita.

### Experiment 2.2
Zvysenie learning rate na 2.0 a predlzenie treningu na 3000 epoch.

In [ ]:
print("\n=== Experiment 2.2 (lr=2.0, 3000 epoch) ===\n")
torch.manual_seed(42)
np.random.seed(42)
model_2_2 = NetMedium()
hist_2_2 = train(model_2_2, data, epochs=3000, lr=2.0, print_every=1000)
s_2_2, a_2_2, r_2_2 = test(model_2_2, data)

**Diskusia 2.2:**
Vyssi lr a viac epoch zlepsilo vysledky. NetMedium ma dostatok kapacity na naucenie vsetkych znakov.

### Experiment 2.3
Krokovy learning rate - rovnaky princip ako v Model 1.

In [ ]:
# [AI] napad krokoveho lr som nasiel s pomocou AI
print("\n=== Experiment 2.3 (krokovy lr) ===\n")
torch.manual_seed(42)
np.random.seed(42)
model_2_3 = NetMedium()
hist_2_3 = []
print("--- faza 1: lr=2.0, 1000 epoch ---")
h = train(model_2_3, data, epochs=1000, lr=2.0, print_every=500)
hist_2_3.extend(h)
print("\n--- faza 2: lr=0.5, 1000 epoch ---")
h = train(model_2_3, data, epochs=1000, lr=0.5, print_every=500)
hist_2_3.extend(h)
print("\n--- faza 3: lr=0.05, 500 epoch ---")
h = train(model_2_3, data, epochs=500, lr=0.05, print_every=250)
hist_2_3.extend(h)
s_2_3, a_2_3, r_2_3 = test(model_2_3, data)

**Diskusia 2.3:**
Krokovy lr opat dava najlepsie vysledky. NetMedium s krokovym lr dosahuje vyssiu accuracy a reliability ako NetSmall.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(hist_2_1, label="Exp 2.1 (lr=1.0)")
plt.plot(hist_2_2, label="Exp 2.2 (lr=2.0)")
plt.plot(hist_2_3, label="Exp 2.3 (krokovy lr)")
plt.xlabel("Epocha")
plt.ylabel("Global error")
plt.title("Model 2 (NetMedium) - Priebeh chyby")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.show()

## Model 3: NetLarge (112->256->128->64->32->8)
Vacsia siet s 4 skrytymi vrstvami a ReLU aktivaciou. Vacsi pocet parametrov pre zlozitejsie vzory.

### Experiment 3.1
Zakladny experiment s lr=0.5 a 3000 epochami. Mensi lr pre vacsiu siet.

In [ ]:
print("\n=== Experiment 3.1 (lr=0.5, 3000 epoch) ===\n")
torch.manual_seed(42)
np.random.seed(42)
model_3_1 = NetLarge()
hist_3_1 = train(model_3_1, data, epochs=3000, lr=0.5, print_every=1000)
s_3_1, a_3_1, r_3_1 = test(model_3_1, data)

**Diskusia 3.1:**
NetLarge s lr=0.5 ma pomalsie ucenie, ale vdaka vacsej kapacite dokaze lepsie zachytit vzory v datach. ReLU aktivacia umoznuje rychlejsi prechod gradientov.

### Experiment 3.2
Zvysenie lr na 2.0 a 3000 epoch.

In [ ]:
print("\n=== Experiment 3.2 (lr=2.0, 3000 epoch) ===\n")
torch.manual_seed(42)
np.random.seed(42)
model_3_2 = NetLarge()
hist_3_2 = train(model_3_2, data, epochs=3000, lr=2.0, print_every=1000)
s_3_2, a_3_2, r_3_2 = test(model_3_2, data)

**Diskusia 3.2:**
Vyssi lr na 2.0 zrychlil ucenie. NetLarge s ReLU aktivaciou a vacsim poctom parametrov dosahuje najlepsie vysledky z doteraz testovanych.

### Experiment 3.3
Krokovy learning rate pre NetLarge.

In [ ]:
# [AI] napad krokoveho lr som nasiel s pomocou AI
print("\n=== Experiment 3.3 (krokovy lr) ===\n")
torch.manual_seed(42)
np.random.seed(42)
model_3_3 = NetLarge()
hist_3_3 = []
print("--- faza 1: lr=2.0, 1000 epoch ---")
h = train(model_3_3, data, epochs=1000, lr=2.0, print_every=500)
hist_3_3.extend(h)
print("\n--- faza 2: lr=0.5, 1000 epoch ---")
h = train(model_3_3, data, epochs=1000, lr=0.5, print_every=500)
hist_3_3.extend(h)
print("\n--- faza 3: lr=0.05, 500 epoch ---")
h = train(model_3_3, data, epochs=500, lr=0.05, print_every=250)
hist_3_3.extend(h)
s_3_3, a_3_3, r_3_3 = test(model_3_3, data)

**Diskusia 3.3:**
Krokovy lr pre NetLarge dava najlepsie vysledky zo vsetkych experimentov. ReLU aktivacia v spojeni s krokovym lr umoznuje efektivne ucenie.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(hist_3_1, label="Exp 3.1 (lr=0.5)")
plt.plot(hist_3_2, label="Exp 3.2 (lr=2.0)")
plt.plot(hist_3_3, label="Exp 3.3 (krokovy lr)")
plt.xlabel("Epocha")
plt.ylabel("Global error")
plt.title("Model 3 (NetLarge) - Priebeh chyby")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.show()

## Ulozenie modelov

In [ ]:
os.makedirs("models", exist_ok=True)
torch.save(model_1_1.state_dict(), "models/model1_exp1.pth")
torch.save(model_1_2.state_dict(), "models/model1_exp2.pth")
torch.save(model_1_3.state_dict(), "models/model1_exp3.pth")
torch.save(model_2_1.state_dict(), "models/model2_exp1.pth")
torch.save(model_2_2.state_dict(), "models/model2_exp2.pth")
torch.save(model_2_3.state_dict(), "models/model2_exp3.pth")
torch.save(model_3_1.state_dict(), "models/model3_exp1.pth")
torch.save(model_3_2.state_dict(), "models/model3_exp2.pth")
torch.save(model_3_3.state_dict(), "models/model3_exp3.pth")
print("\nModely ulozene do priecinka models/")

## Test na realnom obrazku
Najlepsie modely z kazdej architektury sa otestuju na realnom obrazku (test-photo.png). Obrazok sa rozreze na bloky 8x14 pixelov a kazdy blok sa klasifikuje ako jeden ASCII znak.

In [ ]:
print("\n" + "=" * 60)
print("TEST NA REALNOM OBRAZKU")
print("=" * 60)
cesta = "test-photo.png"
best_models = [
    ("NetSmall (exp 1.3)", model_1_3),
    ("NetMedium (exp 2.3)", model_2_3),
    ("NetLarge (exp 3.3)", model_3_3),
]
for meno, model in best_models:
    print(f"\n--- {meno} ---")
    ascii_art = obrazok_na_ascii(model, cesta, SIRKA_BLOKU, VYSKA_BLOKU)
    print(ascii_art)
    print()

## Tabulka parametrov

| Model | Architektura | Aktivacia | Parametre | Experiment | LR | Epochy |
|-------|-------------|-----------|-----------|------------|----| -------|
| NetSmall | 112-64-32-8 | Sigmoid | ~8 000 | 1.1 | 1.0 | 2000 |
| NetSmall | 112-64-32-8 | Sigmoid | ~8 000 | 1.2 | 2.0 | 3000 |
| NetSmall | 112-64-32-8 | Sigmoid | ~8 000 | 1.3 | 2.0->0.5->0.05 | 2500 |
| NetMedium | 112-128-64-32-8 | Sigmoid | ~25 000 | 2.1 | 1.0 | 2000 |
| NetMedium | 112-128-64-32-8 | Sigmoid | ~25 000 | 2.2 | 2.0 | 3000 |
| NetMedium | 112-128-64-32-8 | Sigmoid | ~25 000 | 2.3 | 2.0->0.5->0.05 | 2500 |
| NetLarge | 112-256-128-64-32-8 | ReLU+Sigmoid | ~65 000 | 3.1 | 0.5 | 3000 |
| NetLarge | 112-256-128-64-32-8 | ReLU+Sigmoid | ~65 000 | 3.2 | 2.0 | 3000 |
| NetLarge | 112-256-128-64-32-8 | ReLU+Sigmoid | ~65 000 | 3.3 | 2.0->0.5->0.05 | 2500 |

## Zhrnutie

### Model 1 (NetSmall)
Mensia siet s 2 skrytymi vrstvami. Dosahuje priemerne vysledky, ma obmedzenu kapacitu na naucenie sa vsetkych 8 znakov. Krokovy lr zlepsuje vysledky.

### Model 2 (NetMedium)
Stredna siet s 3 skrytymi vrstvami. Lepsia reprezentacna kapacita, vyssia accuracy a reliability. Krokovy lr opat najlepsi.

### Model 3 (NetLarge)
Najvacsi model s 4 skrytymi vrstvami a ReLU aktivaciou. Najlepsie vysledky v testoch aj na realnom obrazku. ReLU umoznuje rychlejsie ucenie.

### Co som zistil:
- Vacsi model = lepsia schopnost generalizacie
- Krokovy lr je najlepsia strategia pre vsetky architektury
- ReLU v hlbsej sieti pomaha rychlejsiemu uceniu oproti Sigmoid
- Pri realnom obrazku najlepsi vysledok dava NetLarge s krokovym lr